In [1]:
import tenseal as ts
import utilities as util

# Key Generation

In [2]:
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)

In [3]:
context.generate_galois_keys() # Create the public and private key pair
context.global_scale = 2**40

In [4]:
secret_context = context.serialize(save_secret_key = True) # Returns a binary output that contains both the private(decryption) and public(encryption) keys
util.write_data('keys/secret.txt', secret_context) # Store the binary output in a text file(by converting it into base64 ascii representation)

In [5]:
# EXTRA: To extract only the public key from the context
context.make_context_public() # Drop the private key from the context, only keep the public key
public_context = context.serialize() # Extract the public key
util.write_data('keys/public.txt', public_context) # Store the public key

# Encryption

In [6]:
data = [10000]

In [7]:
# Load the public key and encrypt the data
public_context = ts.context_from(util.read_data('keys/public.txt')) # public_context now holds the public key

In [8]:
# Encrypt the data
data_encrypted = ts.ckks_vector(public_context, data)

In [9]:
util.write_data('data/data_encrypted.txt', data_encrypted.serialize())

# Decryption

In [20]:
# Perform this after you have done calculations on the encrypted and saved data
# First, load the secret key
context = ts.context_from(util.read_data('keys/secret.txt'))

# Then, load the encrypted data
data_proto = util.read_data('data/data_encrypted_with_calc.txt')
data = ts.lazy_ckks_vector_from(data_proto)
data.link_context(context)

In [21]:
data.decrypt()[0] # Actual calculation was: 10000 * 1.2 + 1000

13000.001608935683

# Extras
**Homomorphic Encryption and Decryption on PyTorch Tensors**

In [22]:
import torch 

In [43]:
data_tensor = torch.tensor([[1.5, 2.5], [5.5, 10.65]]) 
original_shape = data_tensor.shape
print(data_tensor)

tensor([[ 1.5000,  2.5000],
        [ 5.5000, 10.6500]])


In [44]:
data_tensor = data_tensor.flatten()
data_tensor.shape
print(data_tensor)

tensor([ 1.5000,  2.5000,  5.5000, 10.6500])


**1. Encryption**

In [47]:
# Load the public and secret key
public_context = ts.context_from(util.read_data('keys/public.txt'))
secret_context = ts.context_from(util.read_data('keys/secret.txt'))

In [48]:
data_encrypted = ts.ckks_vector(public_context, data_tensor)

**2. Calculations**

In [49]:
data_encrypted *= 10 # Multiply each element by 10
data_encrypted *= (1/2) # Divide each element by 2

**3. Decryption**

In [62]:
data_encrypted.link_context(secret_context)

In [63]:
data_decrypted = data_encrypted.decrypt()
data_decrypted

[7.500006036966943, 12.50001005599656, 27.500022129474683, 53.25004094107456]

In [64]:
# Reshaping the decrypted data into original tensor's shape
data_decrypted = torch.tensor(data_decrypted, dtype=torch.float32).view(original_shape)
data_decrypted

tensor([[ 7.5000, 12.5000],
        [27.5000, 53.2500]])